# Raw data grab: every upstream source of every fire, one folder per fire

Downloads all channels of upstream data behind the wildfire spread dataset
([Javyn123/wildfire-spread-2014-2025](https://huggingface.co/datasets/Javyn123/wildfire-spread-2014-2025)) into a
Hugging Face dataset repo you choose. Each fire ends up with one folder:

```
fires/<fire_id>/
  LINKS.md, links.json    every channel of this fire: status, links to its files here, and the upstream URLs
  history/  perimeters/  firms/  gridmet/  ics209/  suppression/  tessera/  terrain/
  landfire/  lulc/  landcover/  vegetation/  goes/  hrrr/  hrrr_next/  gfs/  osm/  viirs/
```

Each channel holds that source's own records, values and grid, cut to the fire's region and time window. The region
is the box around all its perimeters pushed out 32 km on every side (any 64 km tile centred in the fire fits); the window
is the fire's perimeter days stretched over the FIRMS activity that leads into and out of them, plus a day on
each side: the notebook runs `firms` first, then writes `catalog/fire_extents.csv` (window, ignition point, region), which the time-dependent
channels read. The code is
`pipeline/raw_grab.py` on the `data_grab_satfire` branch of
[WildfireRepo1](https://github.com/GT-Fire-Research-and-Innovation/WildfireRepo1).

**Before you run**
- Settings → Internet: **on**.
- Add-ons → Secrets:
  - `HF_TOKEN_ALL`: a Hugging Face token with write access (the label is set in `SECRET_LABELS`)
  - `FIRMS_MAP_KEY`: free at https://firms.modaps.eosdis.nasa.gov/api/map_key/
  - `EARTHDATA_TOKEN`: free at https://urs.earthdata.nasa.gov (Generate Token, valid 60 days); needed by
    `viirs`, `vegetation` and `landcover`, which are skipped without it
  - `GEE_SERVICE_ACCOUNT_KEY`: the JSON key of a Google Cloud service account with Earth Engine access, and
    `GEE_PROJECT`: that Cloud project's id (register it at https://code.earthengine.google.com/register,
    free for research); needed by `gfs`, which is skipped without them

**How it runs**
- *Run all* goes through every channel, cheapest first. For each channel it fetches every fire, uploads them in
  batches of about 2 GB, and deletes each batch locally once it is uploaded.
- It works channel by channel, not fire by fire, so each big shared download happens once (an OSM snapshot is
  2–3 GB, and one GOES or HRRR file covers many fires). The repo still ends up organised by fire.
- At the end of the session it rebuilds every fire's `LINKS.md` / `links.json`.
- `lulc`, `landcover`, `vegetation`, `hrrr_next`, `viirs` and gridmet's `pr` / `tmmn` / `sph` / `pdsi` add every
  input TS-SatFire has. On a repo that already has gridmet, the next run fetches gridmet again only for the fires
  that miss the new variables.
- A full pull takes several 12-hour sessions, mostly `goes`, `hrrr`, `hrrr_next`, `osm` and `viirs`. Just run the notebook again: fires
  already in the repo's manifests are skipped, and each session picks up where the last one stopped.

In [ ]:
# ---- settings
HF_REPO = "your-name/wildfire-raw-2014-2025"   # target dataset repo (created if missing)
PRIVATE = True                                 # only used when the repo is created
FIRES = "built"                                # built (1,841 fires with a built folder) | keep | all
WORK = "/tmp/raw"                              # scratch: staging, batches, one-time downloads (not /kaggle/working)
MAX_HOURS = 11.5                               # whole session, under Kaggle's 12 h limit
LINKS_RESERVE_H = 0.25                         # time kept for the per-fire links at the end
BATCH_GB = 2.0

CHANNELS = ["history", "perimeters", "firms", "gridmet", "ics209", "suppression", "tessera", "terrain",
            "landfire", "lulc", "landcover", "vegetation", "goes", "hrrr", "hrrr_next", "gfs", "osm", "viirs"]
RUN = CHANNELS                                 # or a subset, e.g. ["firms", "goes"]

# Kaggle secret label (Add-ons > Secrets) for each value the script reads: left = what the script uses,
# right = the secret's label in Kaggle. Change the right side if your secrets are named differently.
SECRET_LABELS = {"HF_TOKEN": "HF_TOKEN_ALL", "FIRMS_MAP_KEY": "FIRMS_MAP_KEY", "EARTHDATA_TOKEN": "EARTHDATA_TOKEN",
                 "GEE_SERVICE_ACCOUNT_KEY": "GEE_SERVICE_ACCOUNT_KEY", "GEE_PROJECT": "GEE_PROJECT"}
BRANCH = "data_grab_satfire"                   # WildfireRepo1 branch the code comes from

In [ ]:
# ---- the project code, BRANCH (settings cell)
!git clone -q --depth 1 -b {BRANCH} https://github.com/GT-Fire-Research-and-Innovation/WildfireRepo1.git 2>/dev/null || (cd WildfireRepo1 && git fetch -q --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD)
!cd WildfireRepo1 && git log -1 --format='%h %s'

In [ ]:
# ---- packages. netCDF4 is upgraded on purpose: Kaggle's preinstalled build failed to open remote data,
# the PyPI wheels read OPeNDAP, HTTPS and in-memory files (tested)
!pip install -q -U netCDF4 pyogrio pygrib h5py pyhdf earthengine-api "huggingface_hub>=0.24"
import shutil
if "osm" in RUN and not shutil.which("osmium"):
    !apt-get -qq update && apt-get -qq install -y osmium-tool aria2 > /dev/null

In [ ]:
# ---- secrets -> the environment variables the script reads (values never printed)
import os
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _env, _label in SECRET_LABELS.items():
        try:
            os.environ[_env] = _s.get_secret(_label)
        except Exception as _e:
            print(f"Kaggle secret {_label!r} not readable ({type(_e).__name__}): add it under Add-ons > Secrets "
                  f"with exactly this label and switch it on for this notebook (or change SECRET_LABELS)")
except ImportError:
    pass                                       # not on Kaggle: use the existing environment
print({env: ("set" if os.environ.get(env) else "MISSING") for env in SECRET_LABELS})
if not os.environ.get("FIRMS_MAP_KEY") and "firms" in RUN:
    print("no FIRMS_MAP_KEY: skipping firms"); RUN = [c for c in RUN if c != "firms"]
_EARTHDATA = {"viirs", "vegetation", "landcover"}
if not os.environ.get("EARTHDATA_TOKEN") and _EARTHDATA & set(RUN):
    print(f"no EARTHDATA_TOKEN: skipping {sorted(_EARTHDATA & set(RUN))}"); RUN = [c for c in RUN if c not in _EARTHDATA]
if not (os.environ.get("GEE_SERVICE_ACCOUNT_KEY") and os.environ.get("GEE_PROJECT")) and "gfs" in RUN:
    print("no GEE_SERVICE_ACCOUNT_KEY / GEE_PROJECT: skipping gfs"); RUN = [c for c in RUN if c != "gfs"]

In [ ]:
import subprocess, sys, time

SCRIPT = "WildfireRepo1/pipeline/raw_grab.py"

def _run(args):
    p = subprocess.Popen([sys.executable, "-u", SCRIPT, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def _opts(opts):
    out = []
    for k, v in opts.items():
        flag = "--" + k.replace("_", "-")
        if v is True:
            out.append(flag)
        elif isinstance(v, (list, tuple)):
            out += [flag, *map(str, v)]
        elif v not in (None, False):
            out += [flag, str(v)]
    return out

def grab(channel, **opts):
    """One channel for every fire (resumes from the repo's manifests). opts: ids=[...], limit=N, dry_run=True, max_hours=..."""
    args = ["--channel", channel, "--hf-repo", HF_REPO, "--work", WORK, "--fires", opts.pop("fires", FIRES),
            "--batch-gb", str(opts.pop("batch_gb", BATCH_GB))] + (["--private"] if PRIVATE else [])
    return _run(args + _opts(opts))

def links(**opts):
    """Rebuild fires/<id>/LINKS.md + links.json and catalog/raw_index.csv from the manifests."""
    return _run(["--links", "--hf-repo", HF_REPO, "--work", WORK, "--fires", opts.pop("fires", FIRES)] + _opts(opts))

def extents(**opts):
    """catalog/fire_extents.csv from the firms files: each fire's window, ignition point and region."""
    return _run(["--extents", "--hf-repo", HF_REPO, "--work", WORK, "--fires", opts.pop("fires", FIRES)] + _opts(opts))

TIMED = {"gridmet", "goes", "hrrr", "hrrr_next", "gfs", "vegetation", "viirs"}   # read catalog/fire_extents.csv

def status():
    return _run(["--status", "--hf-repo", HF_REPO])

In [ ]:
# ---- stop here unless the token can write to HF_REPO (prints exactly what to fix)
assert _run(["--check", "--hf-repo", HF_REPO]) == 0, "fix the setup above, then rerun from the secrets cell"

Optional check first: every channel for one fire, nothing uploaded. Camp Fire 2018 is `F2003403265`.

In [ ]:
# for ch in RUN:
#     grab(ch, ids=["F2003403265"], dry_run=True)
# !find /tmp/raw/batch -type f | sort | head -40

## Download all channels

In [ ]:
t0 = time.time()
extents_done = False
for ch in RUN:
    if ch in TIMED and not extents_done:          # after firms, before the first channel that needs the windows
        print("\n========== fire extents: window, ignition, region (from firms) ==========")
        extents()
        extents_done = True
    left = MAX_HOURS - LINKS_RESERVE_H - (time.time() - t0) / 3600
    if left < 0.25:
        print(f"session time used up before {ch}: run the notebook again to continue")
        break
    print(f"\n========== {ch} ({left:.1f} h left) ==========")
    rc = grab(ch, max_hours=round(left, 2))
    if rc == 3:                                    # setup error (token / repo): other channels would fail the same way
        print("stopping: fix the setup error above, then rerun")
        break
    if rc == 4:                                    # this source failed for many fires in a row: try the others
        print(f"{ch} stopped early (see CHANNEL STOPPED above); its fires are retried on the next run")

## Per-fire links, then progress

In [ ]:
links()
status()